Primary Test notbook, diarization and whisper. Single file test first.

In [1]:
import whisper
from pyannote.audio import Pipeline
import csv
import ffmpeg
import pydub
from pydub import AudioSegment
import os
import torch
from helper_functions import *
import time
from datetime import datetime


c:\Users\pisces2\Documents\Audio_Transcription_Deidentification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Folder locations: Local only (testing)

In [8]:
#Start loop through each file in untranscribed_audio:

project_path = 'C://Users//pisces2//Documents//Audio_Transcription_Deidentification//'
new_audio_files = project_path + 'untranscribed_audio//'
transcription_output_folder = project_path + 'transcripts//'
temp_audio_clips_folder = project_path + 'audio_clips//'

rttm_folder  = project_path + 'rttm_files//'

wav_folder =  project_path + 'wav_files_unt//'

transcribed_audio_folder = project_path + 'transcribed_audio//'

Folder Locations: Box (project location)

In [2]:
project_path = 'C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//'
new_audio_files = project_path + 'audio_files//'

#Where my Whisper Transcription goes
transcription_output_folder = project_path + 'transcription_train//'

#Where my pre-transcribed file is taken from
transcription_true = project_path + 'transcription_test//'
temp_audio_clips_folder = project_path + 'audio_clips//'

rttm_folder  = project_path + 'rttm_files//'

wav_folder =  project_path + 'wav_files//'

transcribed_audio_folder = project_path + 'finished_audio//'

pyan_audio_clips = project_path + "pyan_clips//"

Create diarization and rttm file. Group lines by speaker. 

In [3]:
new_audio =  os.listdir(new_audio_files)
print(new_audio)

#new_audio_files = "C://Users//pisces2//Documents//Audio_Transcription_Deidentification//untranscribed_audio//"
#new_audio = 'hamlet_test.m4a'

new_audio = 'PC1-1025-01_S1_2022.02.28.m4a'

init_aud_file = os.path.join(new_audio_files, new_audio)
#init_aud_file = new_audio

print("File is", init_aud_file)
#audio file without extentions:
#aud =  os.path.splitext(new_audio)[0]
#aud = 'hamlet_test'
aud = 'PC1-1025-01'
#aud = new_audio
# checking if it is a file
if os.path.isfile(init_aud_file):
    print("Working on:" , init_aud_file)

#create output file variables:
new_wav_file = aud + ".wav"
print("new_wav_file", new_wav_file)
new_rttm_file = aud + ".rttm"
print("new_rttm", new_rttm_file)
transcript_output = aud + ".csv"
print("Transcript output", transcript_output)

convert_to_wav(init_aud_file, new_wav_file)
print("converted to wav")

audio = AudioSegment.from_file(init_aud_file)
audio = audio.set_frame_rate(16000)

start_trim = milliseconds_until_sound(audio)
trimmed = audio[start_trim:]
print(" trimmed complete")
export_path = os.path.join(wav_folder, new_wav_file)

#Segment Audio:
segment_length = 5 * 60 * 1000 

trimmed.export(export_path, format='wav')

first_five_min = trimmed[: 5 * 60 * 1000]
first_five_min.export(wav_folder + "first_five_min.wav", format= "wav")

print("all complete")

['PC1-1025-01_S1_2022.02.28.m4a']
File is C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_files//PC1-1025-01_S1_2022.02.28.m4a
Working on: C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_files//PC1-1025-01_S1_2022.02.28.m4a
new_wav_file PC1-1025-01.wav
new_rttm PC1-1025-01.rttm
Transcript output PC1-1025-01.csv
converted to wav
 trimmed complete
all complete


PIANNOTE- Diarization

Splitting audio for pyannote: 

- dont need to worry about splitting in mid-speech. Pyannote is about voice activity detection and speaker separation. Splitting mid-speech will be corrected later when speakers are combined. 
- WILL need to ensure that each separate audio file starting from the first maintains the same speaker annotation. (speaker 0 does not become speaker 1 accross audio clips)

    - Save initial speaker segment as part of pre-trained? First 5 min segment (or 8-10, to ensure both speakers)

In [39]:
# Segment audio for pyannote : for faster processing.
import numpy as np
from scipy.io.wavfile import write

def downsample_audio(input_path , output_path, target_sample_rate=16000):
    audio = AudioSegment.from_file(input_path)

    audio = audio.set_frame_rate(target_sample_rate)

    audio.export(output_path, format='wav')


def split_audio(input_path, segment_duration=300, output_dir=temp_audio_clips_folder):
    import os
    os.makedirs(output_dir, exist_ok=True)

    audio= AudioSegment.from_file(input_path)

    total_duration = len(audio) / 1000

    num_segments = int(np.ceil(total_duration / segment_duration))

    for i in range(num_segments):
        start_time = i * segment_duration * 1000
        end_time = min((i + 1) * segment_duration * 1000, len(audio))

        segment = audio[start_time:end_time]

        segment_path = f"{output_dir}/segment_{i + 1}.wav"

        segment.export(segment_path, format = 'wav')


input_audio = wav_folder + "first_five_min.wav"
downsampled_audio = wav_folder + "downsampled_fiv_min.wav"

downsample_audio(input_audio, downsampled_audio)

split_audio(downsampled_audio, segment_duration=300, output_dir= temp_audio_clips_folder)


In [ ]:
# Optimized Diarization

from pyannote.audio.pipelines import SpeakerDiarization
from pyannote.pipeline import Optimizer 
from pyannote.audio import Model
from pyannote.audio.tasks import Segmentation


model = Model.from_pretrained("pyannote/segmentation", use_auth_token=True)
task = Segmentation(
    dataset,
    duration=model.specifications.duration,
    max_num_speakers=len(model.specifications.classes),
    batch_size=32,
    num_workers=2,
    loss="bce",
    vad_loss="bce"

)

pipeline = SpeakerDiarization(
    segmentation=finetuned_model,
    clustering='OracleClustering', 

)



In [ ]:
new_audio =  os.listdir(wav_folder)
print(new_audio)
print(new_wav_file)

new_audio = os.listdir(temp_audio_clips_folder)
new_file = temp_audio_clips_folder + new_audio[0]


pre_pyan = time.time()

# instantiate the pipeline
#pipeline = Pipeline.from_pretrained(
 #   "pyannote/speaker-diarization-3.1",
  #  )


#pipeline.params['min_duration_on'] = 0.5
#pipeline.params['min_duration_off'] = 0.3

#new_file = wav_folder + new_wav_file
# run the pipeline on an audio file

device = torch.device("cpu")
pipeline.to(device)


diarization = pipeline(new_file, num_speakers=2)


post_pyan = time.time()

pyan_time = (post_pyan - pre_pyan) / 60
print("Pyannote time", pyan_time)
# dump the diarization output to disk using RTTM format
# make temp file? - dont use specialized naming... 

output_rttm = os.path.join(rttm_folder, new_rttm_file)
with open(output_rttm, "w") as rttm:
    diarization.write_rttm(rttm)

post_rttm = time.time()

post_rttm_file = (post_rttm - post_pyan) / 60
print("Post rttm", post_rttm_file)
print("All time for payn", post_rttm - pre_pyan)
# https://colab.researchgoogle.com/github/Majdoddin/nlp/blob/main/Pyannote_plays_and_whisper_rhymes_v_2_0.ipynb#scrollTo=AMxtBOk4n8IY
print("Post dia / pyannote ")

rttm_file = rttm_folder + new_rttm_file
dlines = open(rttm_file).read().splitlines()
groups = []
g = []
lastend = 0
rtm_length = 0
#group rttm lines by speaker
for d in dlines:

    start_time, end_time, duration = extract_end_time(d)
    rtm_length += end_time - start_time

    #if new speaker, OR if rtm_len > 30
    if g and (g[0].split()[7] != d.split()[7]) or rtm_length > 20:
        groups.append(g)
        g=[]
        rtm_length = 0
    
    g.append(d)


    end_time = end_time * 1000
    if (lastend > end_time):
        groups.append(g)
        g = []
    else:
        lastend = end_time
    
if g:
    groups.append(g)


['downsampled_fiv_min.wav', 'first_five_min.wav', 'PC1-1025-01.wav']
PC1-1025-01.wav


C:\Program Files\Python311\Lib\inspect.py:992: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  if ismodule(module) and hasattr(module, '__file__'):
c:\Users\pisces2\Documents\Audio_Transcription_Deidentification\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:104: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1808.)
  std = sequences.std(dim=-1, correction=1)


Pyannote time 361.0975167751312
Post rttm 0.021754741668701172
All time for payn 361.1192715167999
Post dia / pyannote 


In [31]:

rttm_file = rttm_folder + new_rttm_file
dlines = open(rttm_file).read().splitlines()
groups = []
g = []
lastend = 0
rtm_length = 0
#group rttm lines by speaker
for d in dlines:

    start_time, end_time, duration = extract_end_time(d)
    rtm_length += end_time - start_time

    #if new speaker, OR if rtm_len > 30
    if g and (g[0].split()[7] != d.split()[7]) or rtm_length > 20:
        groups.append(g)
        g=[]
        rtm_length = 0
    
    g.append(d)


    end_time = end_time * 1000
    if (lastend > end_time):
        groups.append(g)
        g = []
    else:
        lastend = end_time
    
if g:
    groups.append(g)


In [32]:
#list of data to add to final transcript output; speaker, start, end
start_stop_list = []
audio = AudioSegment.from_file(file = new_wav_file, format = "wav") + 5
gidx = -1
seconds_count = 0

print("Start time seg list")
    #segment audio file based on speaker turns
for g in groups:
 
    start, nend_time, duration = extract_end_time(g[0])
    nstart_time, end, duration = extract_end_time(g[-1])
    duration = (end - start ) / 1000
    start_time = int(start * 1000)
    end_time = int(end * 1000)
    if (end-start) > 30:

        print("seg time", end-start)
    gidx += 1
    audio_seg = pydub.AudioSegment.empty()

    audio_seg = audio[start_time:end_time]

        #save to folder for temp audio files: to empty after each run. 
    filename =   str(gidx) + ".wav"
    use_name = temp_audio_clips_folder + filename
    print("export to", use_name)
    audio_seg.export(use_name, format ="wav")

    start_stop_list.append([g[0].split()[7], start_time, end_time, duration])

Start time seg list
export to C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//0.wav
export to C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//1.wav
export to C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//2.wav
export to C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//3.wav
export to C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//4.wav
export to C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//5.wav
export to C://Users//pisces2//Box//PIS

In [34]:
print("Start Whisper")
print(start_stop_list)
target_output = transcription_output_folder + transcript_output
print("target out put ", target_output)

with open(target_output, "w", newline='', encoding = "utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Speaker', 'Start Time', 'End Time', 'Transcription'])
    model = whisper.load_model("base")
    for i in range(gidx + 1):
        #get from audio clips foldery
        audiof = temp_audio_clips_folder + str(i) + '.wav'
        print("audio clips from ", audiof)
        
        result = model.transcribe(audio = audiof, language = 'en', word_timestamps=True)
        line_items = start_stop_list[i]
        writer.writerow([line_items[0], line_items[1], line_items[2], result["text"]])
    
print("Wisper complete for ", aud)

#empty temp folder:
for temp_audio in os.listdir(temp_audio_clips_folder):
    full_path = os.path.join(temp_audio_clips_folder, temp_audio)
    if os.path.isfile(full_path):
        os.remove(full_path)
    
print("deleted temp audio")

Start Whisper
[['SPEAKER_01', 31, 639, 0.000608], ['SPEAKER_00', 1634, 2038, 0.0004049999999999998], ['SPEAKER_01', 2410, 3439, 0.001029], ['SPEAKER_00', 4469, 5060, 0.0005910000000000002], ['SPEAKER_01', 4655, 4722, 6.700000000000016e-05], ['SPEAKER_00', 5583, 16433, 0.01085], ['SPEAKER_01', 12012, 12619, 0.0006069999999999994], ['SPEAKER_01', 16771, 17176, 0.0004050000000000011], ['SPEAKER_00', 17176, 18290, 0.0011140000000000008], ['SPEAKER_01', 19133, 20028, 0.0008949999999999995], ['SPEAKER_00', 20028, 20044, 1.699999999999946e-05], ['SPEAKER_01', 20045, 20062, 1.699999999999946e-05], ['SPEAKER_00', 20062, 21817, 0.001754999999999999], ['SPEAKER_01', 21817, 24938, 0.0031209999999999988], ['SPEAKER_00', 24550, 43214, 0.018663999999999997], ['SPEAKER_01', 41898, 42573, 0.0006749999999999971], ['SPEAKER_00', 44092, 46134, 0.0020420000000000017], ['SPEAKER_01', 46758, 50184, 0.003426000000000002], ['SPEAKER_00', 50622, 52090, 0.0014680000000000036], ['SPEAKER_01', 52732, 72812, 0.0200

c:\Users\pisces2\Documents\Audio_Transcription_Deidentification\.venv\Lib\site-packages\whisper\__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = t

audio clips from  C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//0.wav


c:\Users\pisces2\Documents\Audio_Transcription_Deidentification\.venv\Lib\site-packages\whisper\transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


audio clips from  C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//1.wav
audio clips from  C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//2.wav
audio clips from  C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//3.wav
audio clips from  C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//4.wav
audio clips from  C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//5.wav
audio clips from  C://Users//pisces2//Box//PISCES-PC R01 - Operations//Qualitative Data//Rachael - Transcription Project//Transcribed Data//Test//audio_clips//6.wav
audio clip